# Shinkansen Travel Experience Prediction

### Institute Data Science Hackathon — 2nd Place 🥈

**Objective:** Predict whether a passenger was satisfied with their overall travel experience on the Shinkansen Bullet Train.

This notebook documents the solution developed for the institute's Data Science hackathon. The solution combines passenger travel information with survey feedback and uses a **CatBoost + XGBoost ensemble**.

> **Hackathon result:** 2nd place  
> **Recorded validation accuracy during the hackathon:** 95.87%

The notebook has been cleaned and reorganized for GitHub while preserving the original modeling approach.

## 1. Problem Overview

The hackathon provided two related datasets for both training and testing:

- **Travel data** — passenger and journey attributes such as age, travel class, distance, and delays.
- **Survey data** — passenger feedback on comfort, service, cleanliness, entertainment, online services, and other aspects of the journey.

The target variable is:

- `Overall_Experience = 1` → Satisfied
- `Overall_Experience = 0` → Not satisfied

The Travel and Survey datasets are linked using `ID`.

## 2. Project Structure and Data

Place the four competition files in the repository's `data/` folder:

```text
data/
├── Traveldata_train_(1).csv
├── Surveydata_train_(1).csv
├── Traveldata_test_(1).csv
└── Surveydata_test_(1).csv
```

The competition datasets are not redistributed in this repository unless their competition terms permit it.

If your downloaded files have different names, update the four paths in the next cell.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

DATA_DIR = Path("../data")

TRAIN_TRAVEL = DATA_DIR / "Traveldata_train_(1).csv"
TRAIN_SURVEY = DATA_DIR / "Surveydata_train_(1).csv"
TEST_TRAVEL = DATA_DIR / "Traveldata_test_(1).csv"
TEST_SURVEY = DATA_DIR / "Surveydata_test_(1).csv"

required_files = [TRAIN_TRAVEL, TRAIN_SURVEY, TEST_TRAVEL, TEST_SURVEY]
missing_files = [str(p) for p in required_files if not p.exists()]

if missing_files:
    raise FileNotFoundError(
        "Missing competition dataset(s). Place the four CSV files in the data/ folder:\n"
        + "\n".join(missing_files)
    )

travel_train = pd.read_csv(TRAIN_TRAVEL)
survey_train = pd.read_csv(TRAIN_SURVEY)

travel_test = pd.read_csv(TEST_TRAVEL)
survey_test = pd.read_csv(TEST_SURVEY)

train = pd.merge(travel_train, survey_train, on="ID", how="inner")
test = pd.merge(travel_test, survey_test, on="ID", how="inner")

print("Travel train:", travel_train.shape)
print("Survey train:", survey_train.shape)
print("Merged train:", train.shape)
print("Merged test:", test.shape)

## 3. Initial Data Inspection

In [ ]:
display(train.head())
print("\nData types:")
display(train.dtypes)

print("\nTarget distribution:")
display(train["Overall_Experience"].value_counts(normalize=True).sort_index().rename("proportion"))

print("\nMissing values in training data:")
missing_summary = train.isna().sum().sort_values(ascending=False)
display(missing_summary[missing_summary > 0].to_frame("missing_count"))

## 4. Feature Engineering

The original hackathon solution engineered features intended to capture journey disruption and passenger age groups:

1. **Total_Delay** — departure delay + arrival delay.
2. **Delay_per_km** — total delay relative to travel distance.
3. **Age_group** — binned passenger age.

The competition's actual column names are used below (`Departure_Delay_in_Mins` and `Arrival_Delay_in_Mins`).

In [ ]:
def feature_engineering(df):
    df = df.copy()

    departure_delay = df["Departure_Delay_in_Mins"].fillna(0)
    arrival_delay = df["Arrival_Delay_in_Mins"].fillna(0)

    df["Total_Delay"] = departure_delay + arrival_delay
    df["Delay_per_km"] = df["Total_Delay"] / (df["Travel_Distance"] + 1)

    if "Age" in df.columns:
        df["Age_group"] = pd.cut(
            df["Age"],
            bins=[0, 18, 35, 60, 100],
            labels=["0-18", "19-35", "36-60", "61-100"],
            include_lowest=True
        )

    return df

train = feature_engineering(train)
test = feature_engineering(test)

display(train[["Age", "Age_group", "Travel_Distance",
               "Departure_Delay_in_Mins", "Arrival_Delay_in_Mins",
               "Total_Delay", "Delay_per_km"]].head())

## 5. Prepare Features and Target

In [ ]:
X = train.drop(columns=["Overall_Experience", "ID"])
y = train["Overall_Experience"].astype(int)

X_test = test.drop(columns=["ID"])

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=["number"]).columns.tolist()

print("Categorical features:", len(cat_cols))
print(cat_cols)

print("\nNumerical features:", len(num_cols))
print(num_cols)

## 6. Train / Validation Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Validation rows:", X_val.shape[0])

## 7. Prepare Data for CatBoost

CatBoost can work directly with categorical variables. Missing categorical values are converted to an explicit `"Missing"` category so the model receives valid categorical values.

In [ ]:
X_cat = X.copy()
X_test_cat = X_test.copy()

for col in cat_cols:
    X_cat[col] = X_cat[col].astype("object").fillna("Missing").astype(str)
    X_test_cat[col] = X_test_cat[col].astype("object").fillna("Missing").astype(str)

X_train_cat = X_cat.loc[X_train.index]
X_val_cat = X_cat.loc[X_val.index]
X_test_cat = X_test_cat

cat_features = cat_cols

## 8. Prepare Data for XGBoost

XGBoost is given a numeric matrix. Categorical variables are one-hot encoded, with an additional indicator for missing categories.

The same encoded columns are used for training, validation, and test data.

In [ ]:
X_encoded = pd.get_dummies(
    X,
    columns=cat_cols,
    dummy_na=True
)

X_test_encoded = pd.get_dummies(
    X_test,
    columns=cat_cols,
    dummy_na=True
)

# Ensure test has exactly the same columns and order as training.
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

X_encoded = X_encoded.replace([np.inf, -np.inf], np.nan).fillna(-999)
X_test_encoded = X_test_encoded.replace([np.inf, -np.inf], np.nan).fillna(-999)

X_train_encoded = X_encoded.loc[X_train.index]
X_val_encoded = X_encoded.loc[X_val.index]

print("Encoded training shape:", X_train_encoded.shape)

## 9. Model 1 — CatBoost

In [ ]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=600,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_state=42,
    verbose=0
)

cat_model.fit(
    X_train_cat,
    y_train,
    cat_features=cat_features
)

cat_val_prob = cat_model.predict_proba(X_val_cat)[:, 1]

print("CatBoost validation accuracy:",
      accuracy_score(y_val, (cat_val_prob > 0.50).astype(int)))

## 10. Model 2 — XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_model.fit(
    X_train_encoded,
    y_train
)

xgb_val_prob = xgb_model.predict_proba(X_val_encoded)[:, 1]

print("XGBoost validation accuracy:",
      accuracy_score(y_val, (xgb_val_prob > 0.50).astype(int)))

## 11. Ensemble Weight and Threshold Search

The hackathon solution blended the probability predictions from CatBoost and XGBoost.

For each combination:

```text
blended_probability =
    weight × CatBoost_probability
    + (1 - weight) × XGBoost_probability
```

A classification threshold was then searched to convert the blended probability into the final class.

The original hackathon search produced:

- **Best weight:** 0.46
- **Best threshold:** 0.52
- **Validation accuracy:** 95.87%

In [ ]:
from joblib import Parallel, delayed

def evaluate_blend(weight, threshold, cat_prob, xgb_prob, target):
    blended = weight * cat_prob + (1 - weight) * xgb_prob
    prediction = (blended > threshold).astype(int)
    return accuracy_score(target, prediction), weight, threshold

weights = np.arange(0.45, 0.56, 0.01)
thresholds = np.arange(0.47, 0.53, 0.01)

results = Parallel(n_jobs=-1)(
    delayed(evaluate_blend)(
        w, t, cat_val_prob, xgb_val_prob, y_val
    )
    for w in weights
    for t in thresholds
)

best_score, best_weight, best_threshold = max(results, key=lambda x: x[0])

print(f"Best weight: {best_weight:.2f}")
print(f"Best threshold: {best_threshold:.2f}")
print(f"Best validation accuracy: {best_score:.4%}")

### Hackathon result note

The original notebook recorded **95.8678% validation accuracy** at weight **0.46** and threshold **0.52**.

Because the cleaned notebook makes the previously missing preprocessing/model definitions explicit, the locally reproduced score should be treated as a reproducibility check rather than silently assumed to be identical to the original run.

If the rerun gives a slightly different score because of library versions or preprocessing details, retain the original **95.87% hackathon result** as the historical competition-development result.

## 12. Final Ensemble Training

In [ ]:
BEST_WEIGHT = 0.46
BEST_THRESHOLD = 0.52

# CatBoost: train multiple seeds on the complete training set.
cat_test_preds = []

for seed in [42, 52]:
    model = CatBoostClassifier(
        iterations=600,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=5,
        random_state=seed,
        verbose=0
    )

    model.fit(
        X_cat,
        y,
        cat_features=cat_features
    )

    cat_test_preds.append(
        model.predict_proba(X_test_cat)[:, 1]
    )

cat_test_prob = np.mean(cat_test_preds, axis=0)

# XGBoost: train multiple seeds on the complete training set.
def train_and_predict_xgb(seed):
    model = XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        eval_metric="logloss",
        n_jobs=-1
    )

    model.fit(X_encoded, y)

    return model.predict_proba(X_test_encoded)[:, 1]

xgb_test_preds = Parallel(n_jobs=-1)(
    delayed(train_and_predict_xgb)(seed)
    for seed in [42, 52]
)

xgb_test_prob = np.mean(xgb_test_preds, axis=0)

print("CatBoost test predictions:", cat_test_prob.shape)
print("XGBoost test predictions:", xgb_test_prob.shape)

## 13. Generate Final Predictions and Submission

In [ ]:
final_probability = (
    BEST_WEIGHT * cat_test_prob
    + (1 - BEST_WEIGHT) * xgb_test_prob
)

final_prediction = (
    final_probability > BEST_THRESHOLD
).astype(int)

submission = pd.DataFrame({
    "ID": test["ID"],
    "Overall_Experience": final_prediction
})

submission_path = Path("../submission/submission_final.csv")
submission.to_csv(submission_path, index=False)

print("Submission shape:", submission.shape)
display(submission.head())
print(f"Saved to: {submission_path}")

## 14. Solution Summary

### Modeling approach

| Component | Approach |
|---|---|
| Data integration | Merge Travel + Survey data using `ID` |
| Feature engineering | Total delay, delay per km, age groups |
| Model 1 | CatBoost |
| Model 2 | XGBoost |
| Ensemble | Weighted probability blend |
| Weight search | Validation-based grid search |
| Threshold search | Validation-based grid search |
| Final ensemble | Average predictions across two random seeds |
| Evaluation used during hackathon | Accuracy |

### Recorded hackathon result

**🥈 2nd Place**

**Validation accuracy:** **95.87%**

**Best blend weight:** **0.46**

**Best threshold:** **0.52**

## 15. Key Takeaways

- Combining travel attributes with passenger survey feedback provides a strong basis for satisfaction prediction.
- Feature engineering around **journey delays** adds information beyond the raw delay columns.
- CatBoost is useful for the many categorical survey variables.
- XGBoost provides a complementary tree-based model using one-hot encoded features.
- Blending the two models improved the validation result during the hackathon.
- Searching both the **ensemble weight** and **classification threshold** was an important part of the final solution.

## 16. Limitations and Reproducibility

This repository documents a hackathon solution rather than a production deployment.

- Competition datasets are kept outside the repository unless redistribution is explicitly permitted.
- The original hackathon notebook was incomplete; this version reconstructs the missing preprocessing/model setup while preserving the documented modeling strategy.
- The recorded **95.87%** figure is the original hackathon validation result.
- Exact reruns can vary slightly with Python/library versions and preprocessing implementation.

## 17. Future Improvements

- Add systematic cross-validation for ensemble weight selection.
- Compare additional boosting algorithms.
- Perform feature-importance and SHAP analysis.
- Calibrate probabilities before threshold selection.
- Add experiment tracking for model versions and validation results.